## **Best Score on leadboard**

*Setup*

In [ ]:
# Import modules
import numpy as np  # Library for numerical computing and array operations
import pandas as pd  # Library for handling tabular data
import matplotlib.pyplot as plt  # Library for basic data visualization
import seaborn as sns  # Library for statistical data visualization
from sklearn.preprocessing import LabelEncoder  # Encoder for converting categorical variables to numeric labels
from sklearn.ensemble import RandomForestClassifier  # Random Forest classifier
from sklearn.model_selection import StratifiedKFold  # Class for stratified K-fold cross-validation
from sklearn.metrics import roc_auc_score  # Metric function for computing ROC AUC
!pip install catboost

*Load the data*

In [11]:
# If you work with Google Colaboratory, please run this as well.
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# Specify the directory where this notebook is located after %cd.
%cd "/content/drive/MyDrive/GCI_World/competition"

/content/drive/MyDrive/GCI_World/competition


Check path

In [13]:
from pathlib import Path

PATH = Path.cwd() / "input"

train_file = PATH / "train.csv"
test_file = PATH / "test.csv"
sample_sub_file = PATH / "sample_submission.csv"

# Check if path exists
if train_file.exists() and test_file.exists() and sample_sub_file.exists():
    print("All files exist and path is correctly set.")
else:
    print("Some files are missing or path is not correctly set.")

All files exist and path is correctly set.


*Data Overview*

In [14]:
PATH = Path.cwd() / "input"

train = pd.read_csv(PATH / "train.csv")
test = pd.read_csv(PATH / "test.csv")
sample_sub = pd.read_csv(PATH / "sample_submission.csv")

print(f"train: {train.shape}, test: {test.shape}, sample_sub: {sample_sub.shape}")

train: (2781, 16), test: (696, 15), sample_sub: (696, 2)


*Pre-Processing*

In [15]:
# Drop identifier column (not predictive)
train = train.drop(columns="Id")
test = test.drop(columns="Id")


# Frecuency-encode player positions
position_freq = train["Position"].value_counts()
train["Position_Freq"] = train["Position"].map(position_freq)
test["Position_Freq"] = test["Position"].map(position_freq).fillna(0)


# Count the numbers of missing combine drill measurements
combine_cols = [
    "Sprint_40yd",
    "Vertical_Jump",
    "Bench_Press_Reps",
    "Broad_Jump",
    "Agility_3cone",
    "Shuttle"
]

for df in [train, test]:
    df["Total_Missing_Tests"] = (
        df[combine_cols].isna().sum(axis=1)
    )


# Measure athletic performance relative to players at the same position
for med in combine_cols:
  position_median = train.groupby("Position")[med].median()
  for df in [train, test]:
    df[med + "_vs_Position"] = df[med] - df["Position"].map(position_median)


# Measure athletic performance relative to players of the same position group
for med in combine_cols:
    pt_median = train.groupby("Position_Type")[med].median()
    for df in [train, test]:
        df[f"{med}_vs_PositionType"] = (
            df[med]
            - df["Position_Type"].map(pt_median)
        )


# Frecuency-encode schools
school_freq = train["School"].value_counts()
train["School_Freq"] = train["School"].map(school_freq)
test["School_Freq"] = test["School"].map(school_freq).fillna(0)

# Remove original school column after encoding
train = train.drop(columns="School")
test = test.drop(columns="School")


# Numerical combine features to impute
cols_to_fill_mean = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
                     'Broad_Jump', 'Agility_3cone', 'Shuttle']


# Impute missing combine measurements using training-set means
for col in cols_to_fill_mean:
    mean_value = train[col].mean()
    train[col] = train[col].fillna(mean_value)
    test[col] = test[col].fillna(mean_value)


# Create missing age indicator and impute age using the training median
median_age = train["Age"].median()
for df in [train, test]:
    df["Age_Missing"] = df["Age"].isna().astype(int)
    df["Age"] = df["Age"].fillna(median_age)


# Engineer additional athletic performance features
for df in [train, test]:
    # Body Mass Index
    df['BMI'] = (df['Weight'] / (df['Height'] ** 2))

    # Weight-adjusted speed metric
    df['Speed_Score'] = (df['Weight'] / df['Sprint_40yd'])

    # Combined jumping ability
    df["Jump_Total"] = (df["Vertical_Jump"] + df["Broad_Jump"])

    # Upper-body strength realtive to body weight
    df["Bench_per_Weight"] = (df["Bench_Press_Reps"] / df["Weight"])



*CatBoost Predictions*

In [16]:
from catboost import CatBoostClassifier

cat_features = ["Player_Type", "Position_Type", "Position"]

test_pred_proba_list = []
fold_scores = []

X = train.drop(columns=["Drafted"])
y = train["Drafted"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=300,
        loss_function='Logloss',
        eval_metric='AUC',
        verbose=0,
        random_seed=42,
        early_stopping_rounds=100,
        depth=4,
        l2_leaf_reg=5,
        learning_rate=0.05
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set = (X_valid, y_valid)
    )
    valid_pred_proba = model.predict_proba(X_valid)[:, 1]
    score = roc_auc_score(y_valid, valid_pred_proba)
    fold_scores.append(score)
    print(f"Fold {fold + 1} AUC: {score:.4f}")

    test_pred_proba = model.predict_proba(test)[:, 1]
    test_pred_proba_list.append(test_pred_proba)

print(f"Mean CV AUC: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

test_pred_cat = np.mean(test_pred_proba_list, axis=0)

Fold 1 AUC: 0.8072
Fold 2 AUC: 0.8529
Fold 3 AUC: 0.8685
Fold 4 AUC: 0.7991
Fold 5 AUC: 0.8505
Mean CV AUC: 0.8357 ± 0.0274


*Create CSV file*

In [17]:
# Running this cell will generate the submission CSV file
submission = pd.read_csv(PATH / 'sample_submission.csv') # Update PATH in needed
submission["Drafted"] = test_pred_cat
submission.to_csv(PATH / 'submission.csv', index=False)